---
# Chapter 12 — Does Better Context Change Behaviour?

## Orientation

| Field | Value |
|-------|-------|
| Chapter | 12: Does Better Context Change Behaviour? |
| Central question | Does retained history actually change what the system does, and does the change improve the outcome? |
| Main concepts | Behavioural evaluation, Counterfactual test, Memory intervention, Frame-conditioned selection, Assembled context |
| Implementation | behavior_eval (instrument + runner) |
| Experiment | ch12-20260920T204414Z-behavior (frozen behavioural run) |
| Evidence status | Book result |
| Depends on | Chapters 1-11 (all mechanisms), Chapter 2 (instrument) |

---

## What this notebook demonstrates

This is the **behavioural hinge of the book**. The notebook makes the central counterfactual extremely concrete:

```text
same reader
same present task
same evaluation

Condition A: relevant retained past absent
Condition B: relevant retained past present

compare observable behaviour
```

Then distinguishes the causal chain:

```text
history was preserved
history was retrieved
history was selected
history reached context
history was used
behaviour changed
behaviour improved
```

The notebook:

1. **Loads the frozen behavioural run** (`ch12-20260920T204414Z-behavior`)
2. **Shows the matched-condition results** across C0 (no memory), C3 (strong RAG), C4 (assembled context), BA/BR (remove/restore decisive memory)
3. **Displays the token-control runs** (llama and muse) proving the effect is content-specific
4. **Shows real-project transfer** results
5. **Closes the loop through the Memory Measurement Instrument**

> **Evidence status**: Book result. The frozen run is the authoritative book result. A miniature executable example accompanies it but does not replace it.

## The chapter question

> **Does better context change behaviour?**

This is the Chapter 1 counterfactual made runnable: hold present task fixed, vary only the memory intervention, measure whether behaviour changes and whether the change is an improvement.

## Concepts in this chapter

In [ ]:
import sys
from pathlib import Path

def _find_repo_root(start):
    cur = Path(start).resolve()
    while True:
        if ((cur / "content").is_dir() and (cur / "notebooks").is_dir()
                and (cur / "solution").is_dir()):
            return cur
        if cur == cur.parent:
            raise RuntimeError("could not locate repository root")
        cur = cur.parent

REPO_ROOT = _find_repo_root(Path.cwd())
for _p in (str(REPO_ROOT), str(REPO_ROOT / "solution")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from notebooks.memory._support import load_chapter_metadata, render_table

meta = load_chapter_metadata(12)
concepts = (meta.get('chapter', {}).get('concepts')
            or meta.get('concepts', []))
render_table([
    {"Concept ID": c['id'], "Name": c['name'], "Status": c['status']}
    for c in concepts
], "Chapter 12 Concepts")

## The frozen behavioural run

The authoritative book result: `ch12-20260920T204414Z-behavior` — nine controlled tasks (`PLAN` in `behavior_eval/experiments.py`), one fixed reader at temperature zero, task-averaged success. Headline means use the 7-task matched subset (`matched_means`); `corpus-cleanup` and `irrelevant` run reduced condition sets.

In [ ]:
from notebooks.memory._support import load_frozen_run

run = load_frozen_run("ch12-20260920T204414Z-behavior")
summary = run["summary"]

print(f"Run ID: {run['run_id']}")
print(f"Matched task set: {summary['matched_means']['B4']['tasks']}")
print(f"Conditions: {sorted(summary['matched_means'].keys())}")

## The Book Result (from Chapter 18 summary of Chapter 12)

> **Book result** (frozen run, 9 tasks; headline means over the 7-task matched subset, one fixed reader at temperature zero):

| Condition | Description | Task Success |
|-----------|-------------|-------------|
| **B0: No memory** | No supplied history | 0.226 |
| **B1: Full history** | All 68 visible units, 3 projects | 0.048 |
| **B1P: Project-only** | 57 memory-book units | 0.179 |
| **B2: Retrieval-only bundle** | Frozen ch10 C0 bundle | 0.393 |
| **B3: Frame-conditioned** | Frozen ch10 C5 bundle | 0.357 |
| **B4: Assembled** | Frozen ch14 A6@768 render | **0.488** |
| **BO: Oracle** | Evaluator-selected evidence (ceiling) | 0.524 |
| **BA: Remove decisive** | A6@768 minus decisive item | 0.250 |
| **BR: Restore decisive** | BA plus restored item | 0.778 |

Values are the frozen 7-task `matched_means`, rounded; the executable cell below reads them from the artifact.


In [ ]:
# Book-result table read from the frozen artifact, never hand-copied.
labels = {"B0": "B0: No memory", "B1": "B1: Full history",
          "B1P": "B1P: Project-only",
          "B2": "B2: Retrieval-only bundle (frozen C0)",
          "B3": "B3: Frame-conditioned (frozen C5)",
          "B4": "B4: Assembled (A6@768)", "BO": "BO: Oracle ceiling",
          "BA": "BA: Remove decisive", "BR": "BR: Restore decisive"}
render_table(
    [{"Condition": labels[b], "Task Success": m["task_success_macro"],
      "Mean tokens": m["mean_tokens"], "Tasks": m["n_tasks"]}
     for b, m in summary["matched_means"].items() if b in labels],
    "Chapter 12 Book Result (frozen matched_means)")

## Key findings from the book result

1. **Preservation without selection scores BELOW no memory** (0.048 vs 0.226) — the Perfect Memory Paradox arrives as a measurement
2. **Pipeline does not improve monotonically** — 0.393 → 0.357 → 0.488 (framing improves selection metrics without improving behaviour; assembly recovers at 3/5 tokens)
3. **Attribution works when designed for it** — Remove/restore moves same tasks 0.250 → 0.778 (causally helpful, not merely correlated)
4. **Reader-dependent bound** — Two further readers reproduce direction not magnitude (Ministral from zero floor, Muse Spark 1.3 with confirmed 3-repeat ladder)
5. **Negative control worked** — On task whose answer was in current state, both retrieval and frame-conditioned abstained where no-memory scored perfectly

## Load the token-control runs (proves effect is content-specific, not volume)

In [ ]:
# Token-control runs: same budgets, content swapped. Compact view.
for name, run_id in (("Llama", "ch12-token-control-20260921T002019Z-llama"),
                     ("Muse", "ch12-token-control-20260921T003600Z-muse")):
    tc = load_frozen_run(run_id)["summary"]
    print(f"=== Token Control ({name}) ===")
    for cond, stats in tc["mean_by_condition"].items():
        print(f"  {cond}: task_success={stats['task_success']['macro']}, "
              f"mean_tokens={stats['mean_tokens']}")

## Load real-project transfer results

In [ ]:
if 'real-transfer' in run:
    real = run['real-transfer']
    print("=== Real Project Transfer ===")
    print(f"Time-lock commit: {real.get('time_lock_commit')}")
    print(f"Standpoint: {real.get('standpoint')}")
    print(f"Reader: {real.get('reader')}")
    
    tasks = real.get('tasks', {})
    for task_id, conds in tasks.items():
        m0 = conds.get('M0', {}).get('task_score')
        mm = conds.get('Mm', {}).get('task_score')
        delta = (round(mm - m0, 4)
                 if m0 is not None and mm is not None else "N/A")
        print(f"  {task_id}: M0={m0}, Mm={mm}, delta={delta}")

## The capstone RememberingSystem (replays frozen conditions deterministically)

The `capstone` package provides a `RememberingSystem` that replays the frozen conditions without model calls — proving the composed pipeline reproduces the book result.

In [ ]:
from capstone.system import RememberingSystem, load_frozen_conditions
from capstone.runner import REPO, CH12_RUN
from pathlib import Path

ch12_dir = REPO / "experiments" / "benchmark" / "runs" / CH12_RUN
system = RememberingSystem(ch12_dir)

print(f"Loaded frozen conditions: {list(system._frozen.keys())}")

# Run a task through the capstone system
trace = system.run(task_id="fix-store", condition="C4")

print(f"\n=== Capstone Trace for fix-store (C4) ===")
print(f"Task: {trace.task_id}")
print(f"Condition: {trace.condition} ({trace.bcode})")
print(f"History: {trace.history_id}")
print(f"Reader: {trace.reader_id}")
print(f"Budget: {trace.budget_tokens} tokens")
print(f"Retrieved: {trace.retrieved}")
print(f"Selected: {trace.selected}")
print(f"Rejected: {trace.rejected}")
print(f"Context tokens: {trace.context_tokens}")
print(f"Context hash: {trace.context_hash}")
print(f"Behavioural score: {trace.behavioural_score}")
print(f"Harm: {trace.harm}")
print(f"Frame decision: {trace.frame_decision}")
print(f"Trust decisions: {len(trace.trust_decisions)} entries")
print(f"Fallback events: {trace.fallback_events}")

## Run all capstone conditions (replay mode)

In [ ]:
from capstone.conditions import CAPSTONE_CONDITIONS, INTERVENTIONS

print("=== Capstone Conditions ===")
for cap in CAPSTONE_CONDITIONS:
    print(f"  {cap}")
print("\nInterventions:")
for inter in INTERVENTIONS:
    print(f"  {inter}")

# Run a few conditions on one task
for cap in ["C0", "C3", "C4", "BA", "BR"]:
    trace = system.run(task_id="fix-store", condition=cap)
    print(f"\n{cap}: score={trace.behavioural_score}, tokens={trace.context_tokens}, selected={len(trace.selected)}, harm={trace.harm}")

## The measurement instrument closes the loop

After the capstone executes, the observable output is fed into the actual Memory Measurement Instrument. The instrument shows:

- Input task
- Memory condition
- Selected evidence
- Context trace
- Answer/action
- Relevant observations
- Score/failure classifications
- Cost where available

In [ ]:
# Demonstrate the instrument scoring a capstone output
from experiments.benchmark.memory_measurement import MemoryTask, SystemOutput, score_task
from experiments.benchmark.memory_measurement.tasks import ANSWERABLE

# Create a task matching the frozen run
task = MemoryTask(
    task_id="fix-store",
    family="decision",
    prompt="What should new event-store services use?",
    history_ref="ch12-fixture",
    expected_sources=("adr-007",),
    expected_state="PostgreSQL",
    superseded_options=("SQLite",),
    expected_current="PostgreSQL",
    expected_historical="SQLite",
    temporal_mode="current",
    expected_status=ANSWERABLE,
)

# Simulate the C4 output (from frozen trace)
output_c4 = SystemOutput(
    task_id="fix-store",
    answer="New services should use PostgreSQL per adr-007 (July 11 decision). The team chose PostgreSQL instead, after SQLite slowed under concurrent writes.",
    retrieved_ids=("adr-007", "session-035", "session-033"),
    cited_sources=("adr-007",),
    abstained=False,
)

history_ids = ("adr-007", "session-031", "session-033", "session-035", "adr-009", "session-040", "session-044")

print("=== Instrument Scoring of C4 Output ===")
observations = score_task(task, output_c4, history_ids)
for obs in observations:
    val = f"{obs.value:.2f}" if isinstance(obs.value, float) else obs.value
    fail = f" [{obs.failure_class}]" if obs.failure_class else ""
    print(f"  {obs.metric:30s} {val}{fail}")
    if obs.evidence:
        for e in obs.evidence:
            print(f"    → {e}")

## What this establishes

- **Structured, selected, assembled memory changes present behaviour and improves it** against both no-memory floor and strong retrieval baseline
- **The bottom of the ladder is the strongest finding**: Preservation without selection scores below no memory on small reader
- **Pipeline is non-additive**: Framing improves selection without improving behaviour; assembly recovers advantage at ~3/5 tokens
- **Attribution works**: Remove/restore (0.250 → 0.778) with token-matched control showing effect is content-specific
- **Reader-dependent magnitudes**: Direction replicates across 3 readers; magnitudes differ
- **Selection can harm**: On task where answer was in current state, selection abstained where no-memory scored perfectly

## What this does NOT establish

- Broader claim (holds on ordinary work, at scale, across readers) — only 9 controlled tasks
- Of 73 paired comparisons, 57 never reach full success under either condition (tasks are hard, reader is small)
- Which parts of the architecture are load-bearing outside the fixture
- The capstone architecture is a candidate explanation, not the only possible one

## Try it yourself

Explore the remove/restore counterfactual on different tasks. The frozen run contains the decisive memory item IDs for each task — try removing different items and see which ones are actually decisive.

In [ ]:
# TRY IT YOURSELF: inspect the remove/restore memory IDs per task.
# conditions.json maps each B-code to its frozen per-task outcomes.
for bcode in ("BA", "BR"):
    print(f"\n=== {bcode} ===")
    for entry in run["conditions"].get(bcode, []):
        oc = entry.get("outcome", {})
        print(f"  {oc.get('task_id')}: memory_ids={oc.get('memory_ids')}")

## Where this leads next

Chapter 13 resolves the frame hazard: **what happens when the present frame is wrong?** Frame establishment via reconciliation-derived classes matched 9/9 on two readers with no breaches; per-reader simplifications disagree and neither transfers.

> **See this chapter in code:** [Open the companion Jupyter notebook](memory\12-chapter.ipynb)